# PhysioCoach-NLP: retrain Flan-T5 on the paraphrased dataset (Drive-safe)

Free Colab deletes all files when a session resets. This notebook keeps everything important in
your Google Drive, in a folder called **MyDrive/physiocoach**:
- your two zip files are copied there, so you only upload them once
- the trained model is copied there **automatically as soon as training finishes**

**Before you start:** `Runtime -> Change runtime type -> T4 GPU -> Save`.

**First time only:** upload `physiocoach-nlp.zip` and `data_para.zip` with the folder icon on the left.
On later runs, skip the upload: they are already in your Drive.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Connect Google Drive
A window asks you to allow access: choose your Google account and allow.

In [ ]:
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/physiocoach"
os.makedirs(WORK, exist_ok=True)

for f in ["physiocoach-nlp.zip", "data_para.zip"]:
    uploaded, saved = f"/content/{f}", f"{WORK}/{f}"
    if os.path.exists(uploaded):
        shutil.copy(uploaded, saved)
        print(f"{f}: uploaded now, saved a copy to Drive")
    elif os.path.exists(saved):
        print(f"{f}: found in Drive")
    else:
        raise SystemExit(f"{f} is missing: upload it with the folder icon on the left, then run this cell again")

## 3. Unpack the project and the dataset

In [ ]:
import glob, zipfile

with zipfile.ZipFile(f"{WORK}/physiocoach-nlp.zip") as z:
    z.extractall("/content")

# data_para.zip may contain the files directly or inside a data_para/ folder: handle both
shutil.rmtree("/content/_dp", ignore_errors=True)
with zipfile.ZipFile(f"{WORK}/data_para.zip") as z:
    z.extractall("/content/_dp")
train_file = glob.glob("/content/_dp/**/train.jsonl", recursive=True)
assert train_file, "train.jsonl not found inside data_para.zip"
target = "/content/physiocoach-nlp/data_para"
shutil.rmtree(target, ignore_errors=True)
shutil.copytree(os.path.dirname(train_file[0]), target)

os.chdir("/content/physiocoach-nlp")
for name in ["train.jsonl", "val.jsonl", "test.jsonl"]:
    n = sum(1 for _ in open(f"data_para/{name}", encoding="utf-8"))
    print(f"{name}: {n} pairs")

## 4. Install

In [ ]:
!pip install -q -e ".[ml]"

## 5. Train, then save the model to Drive automatically
About 20-40 minutes. You can walk away: the model is copied to **MyDrive/physiocoach/flan-t5-coach-para**
the moment training ends, and the training summary is saved next to it.

In [ ]:
os.chdir("/content/physiocoach-nlp")
!python scripts/train_t5.py --data data_para --model google/flan-t5-small --out models/flan-t5-coach-para --epochs 6 2>&1 | tee /content/train_log.txt

dest = f"{WORK}/flan-t5-coach-para"
shutil.rmtree(dest, ignore_errors=True)
shutil.copytree("models/flan-t5-coach-para", dest, ignore=shutil.ignore_patterns("checkpoints"))
shutil.copy("/content/train_log.txt", f"{WORK}/train_log_para.txt")
print("Saved model to Google Drive:", dest)
print(open(f"{dest}/val_faithfulness.json").read()[:600])

## 6. Quick look at the new model

In [ ]:
!PHYSIOCOACH_T5_DIR=models/flan-t5-coach-para PHYSIOCOACH_TRANSLATOR=none PHYSIOCOACH_TTS=none physiocoach demo -n 8 --backend t5

## 7. Download the model as a zip
If this download fails or the session resets, get it from **drive.google.com -> My Drive -> physiocoach**
instead (right-click the `flan-t5-coach-para` folder -> Download).

In [ ]:
import subprocess
from google.colab import files
os.chdir("/content/physiocoach-nlp")
subprocess.run(["zip", "-qr", "/content/flan-t5-coach-para.zip", "models/flan-t5-coach-para",
                "-x", "models/flan-t5-coach-para/checkpoints/*"], check=True)
shutil.copy("/content/flan-t5-coach-para.zip", f"{WORK}/flan-t5-coach-para.zip")
print("zip size:", round(os.path.getsize("/content/flan-t5-coach-para.zip") / 1e6), "MB (copy also in Drive)")
files.download("/content/flan-t5-coach-para.zip")